# 🎭 페르소나 파인튜닝 — Unsloth (Colab 무료, 텍스트 모델)

> 내 디지털 분신 팀(코라/핀/오피/리나/지오)을 LoRA로 학습.
> **이 노트북 하나로 5명 전부** — 맨 위 `AGENT`만 바꾸면 됨.

**런타임 → 런타임 유형 변경 → T4 GPU** 먼저 선택!

> 💡 모델은 한국어를 잘하는 **텍스트 전용** Qwen2.5-3B 사용 (멀티모달 모델은 형식이 복잡해 제외).
> 위에서부터 **순서대로 ▶** 누르는 게 핵심.

## ① 설정 — 학습할 에이전트 선택

In [ ]:
AGENT = "cora"   # cora | finn | offie | rina | geo
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"  # 한국어 잘하는 텍스트 모델
MAX_SEQ_LEN = 1024
DATA_FILE = f"{AGENT}.jsonl"
print(f"▶ 학습 대상: {AGENT}  |  모델: {MODEL_NAME}")

## ② Unsloth 설치

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir --no-deps unsloth unsloth_zoo

## ③ 모델 로드 (4bit 양자화)

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,
    load_in_4bit = True,
)

## ④ 학습 전 베이스라인 — "아직 코라가 아님" 확인
학습 후 같은 질문을 다시 던져 변화를 비교한다.

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def ask(model, question, max_new_tokens=200):
    FastLanguageModel.for_inference(model)
    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    out = model.generate(
        input_ids=inputs, max_new_tokens=max_new_tokens,
        temperature=0.7, top_p=0.95, top_k=64,
    )
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

print("[학습 전]", ask(model, "넌 누구야?"))

## ⑤ 데이터 업로드 & 형식 변환
실행하면 파일 선택 버튼 → `cora.jsonl` 올리기.

In [ ]:
import os
if not os.path.exists(DATA_FILE):
    from google.colab import files
    print(f"⬆️ {DATA_FILE} 파일을 선택해 업로드하세요")
    files.upload()

from datasets import load_dataset
from unsloth.chat_templates import standardize_data_formats

dataset = load_dataset("json", data_files=DATA_FILE, split="train")
dataset = standardize_data_formats(dataset)
print(f"✅ {len(dataset)}개 로드")

def formatting(examples):
    texts = [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in examples["conversations"]
    ]
    return {"text": texts}

dataset = dataset.map(formatting, batched=True)
print(dataset[0]["text"][:400])

## ⑥ LoRA 어댑터 부착

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## ⑦ 학습 설정
**train_on_responses_only** = 답변(코라의 말)만 학습 → echo 예방.

🎯 목표 **Loss 0.2~0.5**. `>1.0`이면 `max_steps`↑, `<0.01`이면 ↓.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

## ⑧ 학습 🔥 (Loss 보기)

In [ ]:
trainer_stats = trainer.train()
# 마지막 로그의 'loss'가 0.2~0.5 근처인지 확인!

## ⑨ 테스트 🎉 (학습 전과 비교)

In [ ]:
tests = {
    "cora":  ["넌 누구야?", "첫 문장이 안 써져"],
    "finn":  ["넌 누구야?", "AI로 돈 벌기 뭐부터 시작해?"],
    "offie": ["넌 누구야?", "일이 너무 많아서 숨이 막혀"],
    "rina":  ["넌 누구야?", "사람이 잘 안 모여서 속상해"],
    "geo":   ["넌 누구야?", "하고 싶은 게 너무 많아서 뭐부터 해야 할지 모르겠어"],
}
for q in tests.get(AGENT, ["넌 누구야?"]):
    print(f"Q: {q}\nA: {ask(model, q)}\n")

**해석**
- 페르소나 말투/시그니처 나오면 → 🎉 성공
- 질문 echo·밋밋 → 과적합/데이터 부족 → `max_steps`↓ 또는 데이터 50개+
- Loss는 낮은데 말투 X → 형식 점검

## ⑩ 저장 (LoRA zip 다운로드)

In [ ]:
save_dir = f"{AGENT}_lora"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

import shutil
shutil.make_archive(save_dir, "zip", save_dir)
from google.colab import files
files.download(f"{save_dir}.zip")
print(f"💾 저장 완료: {save_dir}.zip")

# (선택) Hugging Face 업로드:
# model.push_to_hub_merged("내아이디/cora", tokenizer, token="hf_...")